In [0]:
from datetime import datetime
import  pyspark.sql.functions as f

dbutils.widgets.text("run_date", "2026-11-04")
run_date = dbutils.widgets.get("run_date")
DROP_DATE = datetime.strptime(run_date, "%Y-%m-%d").date()

DAILY_PATH = (
    f"/Volumes/claim-etl/bronze/landing/healthcare/"
    f"daily/drop_date={DROP_DATE}/"
)
CLAIMS_PATH = DAILY_PATH + "claims/"
MEMBERS_PATH = DAILY_PATH + "members/"
PROVIDERS_PATH = DAILY_PATH + "providers/"

claims_df = spark.read.format("csv") \
    .option("header", "true") \
    .load(CLAIMS_PATH) \
    .withColumns({
        "_source_file": f.col("_metadata.file_path"),
        "_load_date": f.current_date(),
        "_batch_date": f.lit(DROP_DATE)
    })
claims_df.printSchema()
claims_df.limit(10).show()

table_name = "`claim-etl`.bronze.claims"

(
    claims_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("replaceWhere", f"_batch_date = '{DROP_DATE}'")
    .partitionBy("_batch_date")
    .saveAsTable(table_name)
)

In [0]:
members_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(MEMBERS_PATH) \
    .withColumns({
        "_source_file": f.col("_metadata.file_path"),
        "_load_date": f.current_date(),
        "_batch_date": f.lit(run_date)
    })

members_df.limit(10).show()

table_name = "`claim-etl`.bronze.members"

(
    members_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("replaceWhere", f"_batch_date = '{DROP_DATE}'")
    .partitionBy("_batch_date")
    .saveAsTable(table_name)
)

In [0]:
providers_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .load(PROVIDERS_PATH) \
    .withColumns({
        "_source_file": f.col("_metadata.file_path"),
        "_load_date": f.current_date(),
        "_batch_date": f.lit(run_date)
    })
providers_df.limit(10).show(truncate=False)

table_name = "`claim-etl`.bronze.providers"

(
    providers_df.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .option("replaceWhere", f"_batch_date = '{DROP_DATE}'")
    .partitionBy("_batch_date")
    .saveAsTable(table_name)
)

In [0]:
display(spark.table("`claim-etl`.bronze.claims").limit(10))
# spark.sql("DROP TABLE IF EXISTS `claim-etl`.bronze.claims")
display(spark.table("`claim-etl`.bronze.members").limit(10))
display(spark.table("`claim-etl`.bronze.providers").limit(10))

In [0]:
%sql
SELECT _batch_date, COUNT(*) FROM `claim-etl`.bronze.claims GROUP BY 1